### Colab Activity 10.3: Building and Evaluating ARMA Models

The last videos of the module took you through an approach to building ARMA models.  

First, it was important to transform your data into a stationary series if not already so.  You tested for stationarity using the `adfuller` function and interpreted the $p$ value of the hypothesis test.  If the data was not stationary you saw strategies such as differencing and logarithmic transformations applied as a way to achieve stationarity. 

Once the series was stationary, building an ARMA model involved using autocorrelation and partial autocorrelation plots to determine the appropriate $p$ and $q$ parameters of the model.  

This activity asks you to identify a time series of interest and to build an ARMA model to construct a basic forecast for the series and analyze the error.  Another element to consider is to build models with some different $p$ and $q$ models -- as while ACF and PACF plots help us, they are rough ideas of the appropriate parameters, and it is usually good practice to perform a simple grid search on these.  You are to find a time series dataset using any resource you would like and present your model and findings to the class. 

If you have trouble locating a dataset or would prefer a suggestion, try the [Rossmann Store Sales](https://www.kaggle.com/c/rossmann-store-sales) competition from Kaggle.  This involved forecasting retail store sales for a major drugstore.

------

**Further Extensions**

- SARIMA models that extend the ARMA model to include seasonal components.  Statsmodels contains a `SARIMAX` model that will accomplish this. Introduction from docs [here](https://www.statsmodels.org/dev/examples/notebooks/generated/statespace_sarimax_stata.html). 
- ARIMAX and SARIMAX models that incorporate exogenous features as inputs to your model.  The Rossman data provides many features for potential use here. Statsmodels readily implements these.
- Feature Engineering with Time Series.  If you are presented with a basic time series like the sunspots data or airline data without exogenous features, these can be engineered from the series itself.  There are many approaches, but one library that can help is called `tsfresh` docs [here](https://tsfresh.readthedocs.io/en/latest/).

Data was pulled from https://www.kaggle.com/datasets/bobnau/daily-website-visitors

This data set contains information concerning user visits and page loads on a website. The data differentiates from first-time visitors, unique visits and returning visits. My analysis will be focused on forecasting page loads. I will set aside 2020 as our test set.

In [ ]:
from warnings import filterwarnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.forecasting.stl import STLForecast
from statsmodels.tsa.seasonal import seasonal_decompose, STL
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import acf, pacf

filterwarnings("ignore")

In [ ]:
visitors = pd.read_csv("module 10/colab_activity10_2_starter/data/visitors.csv")
visitors.describe()

In [ ]:
visitors["date"] = pd.to_datetime(visitors["Date"])
visitors["Page.Loads"] = visitors["Page.Loads"].str.replace(",", "")
visitors["loads"] = visitors["Page.Loads"].astype("int")
visitors = visitors.drop(
    [
        "Row",
        "Day",
        "Date",
        "Day.Of.Week",
        "Page.Loads",
        "Unique.Visits",
        "First.Time.Visits",
        "First.Time.Visits",
        "Returning.Visits",
    ],
    axis=1,
)
visitors = visitors.set_index("date")

In [ ]:
fig = px.line(visitors.reset_index(), x="date", y="loads", title="Page Loads")
fig.show()

In [ ]:
# Let's save 2020 as test set
y_train = visitors[:-30]
y_test = visitors[-30:]

print(y_train.head())
print(y_train.tail())
print(y_test.head())
print(y_test.tail())

In [ ]:
plt.plot(y_train, label="historical")
plt.plot(y_test, label="future")
plt.grid()
plt.legend();

In [ ]:
# Extract trend
stl = STL(y_train, period=12)
results = stl.fit()

In [ ]:
# plot residuals
plt.plot(results.resid)
plt.grid()
plt.title("Residuals");

In [ ]:
plot_acf(acf(results.resid, nlags=100))

In [ ]:
res = seasonal_decompose(y_train, model="additive", period=30)

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(15, 8))
res.trend.plot(ax=ax1, ylabel="trend")
res.resid.plot(ax=ax2, ylabel="seasonality")
res.seasonal.plot(ax=ax3, ylabel="residual")
plt.show()

In [ ]:
plot_acf(res.seasonal)

In [ ]:
plot_acf(acf(y_train.diff().dropna()))

In [ ]:
# instantiate
stlf = STLForecast(
    y_train,
    ARIMA,
    model_kwargs={"seasonal_order": (1, 0, 1, 6), "freq": "D", "order": (1, 0, 1)},
)
# fit model using historical data
stlf_results = stlf.fit()
# produce forecast for future data
forecast = stlf_results.forecast(len(y_test))

In [ ]:
plt.plot(y_test, label="true future data")
plt.plot(forecast, label="forecast")
plt.plot(y_train, label="training data")
plt.legend()
plt.title("Forecast with STL and Future Data")
plt.grid();

In [ ]:
pred_error = y_test.loads - forecast
mae = np.abs(pred_error).mean()
rmse = np.sqrt(np.square(pred_error).mean())

# Answer check
print(f"MAE: {mae}")
print(f"RMSE: {rmse}")

In [ ]:
plot_acf(acf(y_train.diff().dropna()))
plot_acf(pacf(y_train.diff().dropna()))

In [ ]:
# instantiate
sarimax = SARIMAX(y_train, seasonal_order=(3, 0, 2, 12), freq="D", order=(2, 0, 1))
# fit model using historical data
sarimax_fit = sarimax.fit()
# produce forecast for future data
forecast = sarimax_fit.forecast(len(y_test))

In [ ]:
plt.plot(y_test, label="true future data")
plt.plot(forecast, label="forecast")
plt.plot(y_train, label="training data")
plt.legend()
plt.title("Forecast with STL and Future Data")
plt.grid();

In [ ]:
pred_error = y_test.loads - forecast
mae = np.abs(pred_error).mean()
rmse = np.sqrt(np.square(pred_error).mean())

# Answer check
print(f"MAE: {mae}")
print(f"RMSE: {rmse}")

In [ ]:
def get_model_results(seasonal_order, order):
    # instantiate
    sarimax = SARIMAX(y_train, seasonal_order=seasonal_order, freq="D", order=order)
    # fit model using historical data
    sarimax_fit = sarimax.fit()
    # produce forecast for future data
    forecast = sarimax_fit.forecast(len(y_test))
    pred_error = y_test.loads - forecast
    return np.sqrt(np.square(pred_error).mean())


def find_best_model(seasonal_orders, orders):
    best_rmse = None
    best_season_order = None
    best_order = None
    number_of_fits = len(seasonal_orders) * len(orders)
    fits_ran = 1
    print(f"Number of fits: {number_of_fits}")
    for seasonal_order in seasonal_orders:
        for order in orders:
            print(
                f"Fitting {fits_ran} / {number_of_fits} with season_order: {seasonal_order} order: {order}"
            )
            try:
                fits_ran += 1
                rmse = get_model_results(seasonal_order, order)
                if best_rmse is None or rmse < best_rmse:
                    best_rmse = rmse
                    best_order = order
                    best_season_order = seasonal_order
            except:
                continue

    return best_season_order, best_order, best_rmse

In [ ]:
seasonal_orders = [
    (0, 0, 0, 12),
    (1, 0, 1, 12),
    (1, 0, 0, 12),
    (0, 0, 1, 12),
    (1, 1, 1, 12),
    (2, 0, 2, 12),
    (2, 0, 0, 12),
    (0, 0, 2, 12),
    (2, 1, 2, 12),
]

orders = [
    (0, 0, 0),
    (1, 0, 1),
    (1, 0, 0),
    (0, 0, 1),
    (1, 1, 1),
    (2, 0, 2),
    (2, 0, 0),
    (0, 0, 2),
    (2, 1, 2),
]

best_seasonal_order, best_orders, best_rmse = find_best_model(seasonal_orders, orders)

print(best_seasonal_order, best_orders, best_rmse)

In [ ]:
# instantiate
sarimax = SARIMAX(
    y_train, seasonal_order=best_seasonal_order, freq="D", order=best_orders
)
# fit model using historical data
sarimax_fit = sarimax.fit()
# produce forecast for future data
forecast = sarimax_fit.forecast(len(y_test))
pred_error = y_test.loads - forecast

In [ ]:
plt.plot(y_test, label="true future data")
plt.plot(forecast, label="forecast")
plt.plot(y_train, label="training data")
plt.legend()
plt.title("Forecast with STL and Future Data")
plt.grid();

In [ ]:
pred_error = y_test.loads - forecast
mae = np.abs(pred_error).mean()
rmse = np.sqrt(np.square(pred_error).mean())

# Answer check
print(f"MAE: {mae}")
print(f"RMSE: {rmse}")